# Taller en clase · Calidad de datos: diagnosticar, limpiar y medir
**Ciencia de Datos · Ingeniería Industrial · CORHUILA 2026‑B · Semana 9**

El 26 de septiembre diste tus primeros pasos: cargaste un CSV, lo graficaste y le hiciste una limpieza mínima. Hoy vas un paso más allá. Vas a **medir** la calidad de un registro de la planta, **corregir cada problema con la técnica adecuada**, **dejar constancia** de lo que hiciste y **comprobar con números** que los datos quedaron mejor.

Al terminar sabrás:

1. Diagnosticar la calidad de una tabla con cinco dimensiones: **completitud, unicidad, consistencia, formato y validez**.
2. Quitar **duplicados** y **homologar** textos escritos de varias formas.
3. Convertir **tipos de datos**: números guardados como texto y fechas en dos formatos.
4. Normalizar **unidades** (°F → °C).
5. Decidir entre **eliminar o imputar** un dato faltante.
6. Distinguir un dato **atípico** de un dato **imposible**.
7. Medir la calidad **antes y después** y llevar una **bitácora de limpieza**.
8. Responder la pregunta del negocio con datos confiables.

> **Todo el código ya está escrito y funciona de principio a fin.** Tu trabajo es ejecutar cada celda en orden (**Shift + Enter**), leer el resultado y responder las preguntas. Si algo falla, casi siempre es porque se saltó una celda: usa **Ejecutar todo** (*Run All*) desde el inicio. Si ejecutaste una celda de limpieza dos veces y tus cifras ya no coinciden con los puntos de control, usa **Entorno de ejecución → Reiniciar y ejecutar todo**.

| Símbolo | Qué significa |
|---|---|
| 👉 | La línea clave del paso: la que hace el trabajo importante |
| 🧠 | Pregunta para ti: haz doble clic en la celda de texto que sigue y escribe tu respuesta |
| ✅ | Punto de control: compara tu resultado con el número indicado |

## Parte 0 · Preparar el entorno
Dos celdas dejan todo listo:

1. La primera revisa que estén las librerías **pandas** (tablas) y **matplotlib** (gráficos). En Colab ya vienen; en VS Code las instala sola la primera vez si faltan (tarda un minuto).
2. La segunda **crea el archivo de datos** del taller, `registro_produccion.csv`. Los datos viajan dentro del cuaderno, comprimidos, para que el taller funcione aunque no haya internet. **No tienes que entender esa celda**: solo ejecútala.

In [ ]:
import importlib.util, subprocess, sys

faltan = [lib for lib in ["pandas", "matplotlib"] if importlib.util.find_spec(lib) is None]
if faltan:                                   # solo pasa la primera vez en VS Code
    print("Instalando:", faltan)
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *faltan])

import pandas as pd                 # tablas de datos
import matplotlib.pyplot as plt     # gráficos

plt.rcParams["figure.figsize"] = (9, 4)     # tamaño de los gráficos
pd.set_option("display.max_columns", 20)    # mostrar todas las columnas
print("pandas", pd.__version__, "· listo para trabajar")

In [ ]:
# Crea el archivo registro_produccion.csv a partir de los datos comprimidos (no hace falta entenderla)
import base64, gzip

DATOS = (
    "H4sIAAAAAAACCpVaTY5cNw7e9ykKtVa3RVK/S2MwwGyS2cwFGnYHycJ24rQPMkeZM+RiQ1KqalJPD+gggBGjPkuUSH7kR71f"
    "Xj79+hxef3z/+i18ef7jx29fn8Pv3799/vHpt8/Pf4bPL7+8fHr98e1P/v/Xly+/v3x/ZuzzA0Ysj7E/xhp+ev7rf8/8r356"
    "jBAAC4RA+QlOIBggthyghBMAXWSRGHoP1J/iCSrxMpn/aIHgCR9i/RD7B4GG/zx///wyjYkFA0YxJ9l17hA2hpfh/QKlp7yF"
    "EP/aW4AK4ZoopOsWxZZAK7LXlSgUB/r526dfbwZh7AH4YOTv5w5Rg3pACj0yxBxrQL6oQRBjSHzy8tS2q4g1qHdM8am8Qdri"
    "KyhNTO7JrtMWb1GNAZuYTCcgsYiviG+Imj1XWxyW+FhAYlO3oFfjMIQaEOVkzmzjsF6qLpK9zQMxr6fLBfAVxR1m3g8bbEOw"
    "OTcBMKTKmdMWgmIpqSfBn/gOIYm+HEhXKVsI70HYNYYjx/Ad0teUqkV9idaavriJd5ugcgISixrvlQV0tp3kFaBmBNqTdZ9X"
    "mSFJILSFyAKZ1JP4VLcQtia1HvIaxt27SbytG2ULcSnFjsAUsPszOV/lqL6K/m6sr1JLIWm+1C0khcZcIru8/c4h5h2Vge82"
    "icPzCUgclbsGMDC3Qbzn95JPCTh8PSv5lcRRvJc4KhpHMcg6CgsziRjUtwgxB0Bv2Ka3hYgxHKGZ76by2Y3F1k+xtpkNbiPr"
    "pxI6n9yFjEWImyqzdRk3s4MoFReFxJNVxBQmNT4QNmsKLJ6KHHoIYk1/ANg5QepU6ZpSYH0Oi6cwC10tPofFUxwZozaALSAM"
    "G3d4GUaRkJ/yRT1ipkmEM8vjFiI5zhtx+WQqpgPkcqMd4bfuGYUxPq2ipFWrfhmfVuOmuzfG+Yt34SMdIP/+x7/+eY9iXgUE"
    "Yi4Q10o1/HUCwNC4WxichScYMYYrVKrshRLy9QSWQq95kgVHO67RrlfDpC0uLzbv0PtJODSKK9sRcplpxVUVeMNkacmuw7Zk"
    "UK4ge6x33DD6alZpxnGzJ3JNR1JXMqT8HSq2G/EemW8loZi7t0VXiWKL93Zavc0hPLKTHiCdZKc0msMePFlJWwHhLb5Ha1Fa"
    "s1MYI8vhnU2+keyhCR3jFiE3yAa3hbCTp1H2N+/Eidm2EHZ3F4cqTfSrxVhvxjbKs1CtuRzfQoLW3ujNta4CXsU3CxYgFCGh"
    "x+1htTSa186kxVnk4wlIbOVCJs0UWCbJLDa+8n+XaQ+VpC2t7bj8SlLrUtJ2HuwNZucoSBpflL1FzlOEA+N4NtsWsjOz8fm5"
    "4qXtKmkUxSXS80KgRfp4it7YAbnMq+lNGT/6S3aOqhIyAslbiDaRWViCc8rsVBZXEeuckVP1BCRXU7UCtSa9STnpTVCCuEio"
    "n22n14Nh6Y5h0Wb8dwmLZK+w+B6SyQRpCYo3iCohtraIUCzbRSRoUANZecKcybmq1+mGbFdxKSW1OR7u2DGxeGppii1ENaJ2"
    "bLyKuZZVQXPq6akTnwnqCftJ/mJaCsx9pcu9OORQ45Bl+6XEHlaTNx0Nex0NU2pnb7ereEwFQ0f3LUT7yCJNQ0jA/tztlGaD"
    "AwszeQ0tl9iZJKMjSS+iY67zTHkLoVEVl3TwElr8mUNL1pJ2mHbE2T3XE5Aa0yYB5hOQuIo7E7ngs81U0xe9GwlTaDtH9Tpq"
    "okvNdsipAUlbiJgynOX1M3j9TKA18yruulqQK1V9pPiyjnUUcVHkYN/+rGVBhgrc3vl7cf1+41xZhNmqoDWqaOnku8sW8VNN"
    "2ge4jmKR0J2D71EktLu/fuj3b6MpsiCnzPhYiMNV0L0zbwYRtWO98xoaOLa4N+FKtYdwB98EpnOpeLUYx4C1h4O5jv+ERfvB"
    "FMd/Rdtr5ZEdRHzVddzh+A8XbabFjNdzFOBBGsZd+c9SNh60mSDAV2kDutzKJx56XwTvLA4LXJIGvTLDDgpBc0HolRmyspD2"
    "OJtrthAdkeXZ3KGFWFdRbFMlwxYicSOtOvphBi6yIY1iFU82Ev4TzbUUKzwKs9FqWoWCeGjV0zS5noBojEZJ4jRbdYa4Dj24"
    "8GV/tHcwHKKvZ3xBWHz1RK/gJLF0nGNjB70/Wfpz3nB1xS0kXcTcEtoSFXhIPWnMwF+z86cEe/SDEfTSC7m9I+0295A0Z8/Z"
    "tzBMvc6ffTK2nQZ5jPBkurFb3IHmEDYrcztBpKCvd182brugHNZxiZeLXk7yKXMapnQn0surDQnp3ua4Z7eTTpZ6eOSw0CnX"
    "DiOxx8DVmeSdSSqItj9rhak6iWRj+xZDs55FP8+0EGEJwDky4Qjf6mMY6+CtkzoT0TIGH/MkY88qoqWZkpGH7SpwEdEyqfA6"
    "EL2CTpwIUkAc+VkJLcmSaY476hajYcX5lKTh3y+jfVvyPvD6WV4HJJns7MtCJFxIdUdzzJeWQaQYsVCaV9BQceabyYB8aPaL"
    "GCvVma4WZuSxWNSaFEY3k/dryd0U1PeGtGz48a//fvz54/12irKjqw1eQ1OP85FqD9FHoRvtpy2Ehv5lrmZ23EPSEFVMJN02"
    "Fbio6FJmnY9/i6mzJ1CZU9CSdk5ndxktdD/uxLzogTzHneZ6jzI7Dn+C86fPuwvzLEzq4+02QlssknqIXk4etVvR2cpqk/Nm"
    "GevkEwiOYB4pQVuIxruYotVZzlVOxFs6THqc1OYcT7nNCX7cYsQcbJPR8xZCY8zD6XeVh8nrFqTFl0gzUJ8C2glPYs7z8IZv"
    "VwkHMKYn0QbzQcJpRvjHXVxFHLHlsp07fzN9PxfhNF6trfJ8D4m1Y0O6vBq+L37a4T0h6euPM/itvupcI0995ZaxFqeS9RkZ"
    "/DJOYxSaxbFuIUr/VVpo9ugekubDbF0ueBWEMjITPqVTkD62jGbIke76pJp4ewnGYmO+M327a8bRf5BfydVHWaxpgpnPB3B5"
    "VJVGMAthli1EB7tRfCbinbYYNSfP1wm3jHVXkwBatDAur6pDrI1X1X4crE0BlmBRB335UCGFNt4wKZ61MvH2XPV2JDq8q0pr"
    "lb108qBbm7zMJWl9V6XRBI/26g10cR8qjC7NzY8oLhMxmjU7byE00o+ab4osJA2ma16ckn9Yzdoy6UDMcCG5t9WmUeFfiWh9"
    "Wq1yNVh5J+MJl1q5h2I6U4hiwOKr0uZw/Qwk6ox0etnvOXoAzUiWMlpOV5pDOrBDhwFyBTCPbyzKCQTHGBnkCplVr1uQdAic"
    "ePpRSd0izIAOPcSp91amFO5byKTB8XCQtxAaH1ew+7hbgS1Ev3+h+ShgbFnVe0pa1SmegnA88pgZ3gFyG32T+cDiABKFIC8C"
    "qz3OUaCv5G8suUD0SZ4mJceHiB8g+ibkFjWqM/b7iHySl0PQHi1fLcg3nl0/9yhbAI7Jh8yymt/IvYfKR0ggKdWstb5a6Zn9"
    "KnTIqTqrfX6IdF9n/QCozhblbCWpVsJ/aDq0A0g/5tA6LISSrhbmnm+6kZcbCI45MoKbES8gGopZ2SuHugdpfyFP5fIu1S3E"
    "JVZW1fdGlAvkJhT6mBSYa/QzTNG54g7crqIkWGd3Gt/nsXfEz/8BPDhx+IUpAAA="
)
with open("registro_produccion.csv", "wb") as archivo:
    archivo.write(gzip.decompress(base64.b64decode(DATOS)))

# Un CSV es texto plano: así se ven sus primeras líneas por dentro
with open("registro_produccion.csv", encoding="utf-8") as archivo:
    for _ in range(4):
        print(archivo.readline().rstrip())

✅ **Punto de control:** debes ver la línea de encabezados `fecha,turno,maquina,producidas,defectuosas,temperatura` y tres filas de datos. Cada coma separa una columna; por eso se llama *Comma‑Separated Values*.

## Parte 1 · El caso
La misma planta de empaques del Huila que ya conoces tiene **4 máquinas** en dos líneas de producción y trabaja **3 turnos**. Al final de cada turno, el supervisor digita en una hoja de cálculo compartida cuántas unidades produjo cada máquina, cuántas salieron defectuosas y a qué temperatura cerró la máquina. Hay tres supervisores, cada uno con sus costumbres, y nadie revisa lo que se digita.

| Máquina | Nombre | Línea | Instalada en |
|---|---|---|---|
| M‑01 | Extrusora A | Línea 1 · Película | 2019 |
| M‑02 | Extrusora B | Línea 1 · Película | 2021 |
| M‑03 | Selladora C | Línea 2 · Bolsas | 2012 |
| M‑04 | Cortadora D | Línea 2 · Bolsas | 2023 |

El registro cubre **4 semanas**, del lunes 7 de septiembre al sábado 3 de octubre, de lunes a sábado. La gerencia pregunta:

> **¿Qué máquina y qué turno tienen la mayor tasa de defectos? ¿A cuál le hacemos mantenimiento primero?**

### Diccionario de datos
| Columna | Qué significa | Tipo esperado |
|---|---|---|
| `fecha` | Día del registro, formato `aaaa-mm-dd` | Fecha |
| `turno` | Mañana, Tarde o Noche | Texto (3 valores) |
| `maquina` | Código de la máquina: M‑01 a M‑04 | Texto (4 valores) |
| `producidas` | Unidades fabricadas en el turno | Número entero |
| `defectuosas` | Unidades que no pasaron control de calidad | Número entero |
| `temperatura` | Temperatura de la máquina al cerrar el turno, en °C | Número decimal |

### Reglas del negocio (con ellas vamos a validar)
- La capacidad máxima de una máquina es de **1.800 unidades por turno**.
- Las defectuosas van de **0** hasta el número de unidades producidas.
- El sensor de temperatura mide entre **15 y 60 °C**.
- Solo existen **4 máquinas** y **3 turnos**.

🧠 **Responde:** antes de mirar los datos, ¿qué errores esperas encontrar en una hoja que digitan a mano tres personas distintas durante cuatro semanas? Escribe al menos tres.

*Tu respuesta:* 
Inconsistencia de formato y tipografía:
Cada persona registra los datos con un criterio propio. Por ejemplo, variaciones al ingresar fechas (DD/MM/AAAA vs. MM/DD/AA), diferencias al nombrar un mismo elemento o cliente (abreviaturas, mayúsculas/minúsculas, tildes) o el uso de comas y puntos alternados como separadores decimales.

Errores de digitación (transposición y omisión de dígitos):
Fallos por velocidad de tecleo o cansancio visual, como cambiar el orden de los números (escribir 1234 en lugar de 1243), omitir ceros o ingresar valores en filas/columnas equivocadas.

Registros duplicados o vacíos (datos faltantes):
Falta de coordinación entre los tres turnos u operadores, lo que lleva a registrar la misma operación dos veces o a dejar campos obligatorios en blanco por asumir que otra persona los completaría.



## Parte 2 · Cargar y dar una primera mirada
`pd.read_csv()` lee el archivo y lo convierte en un **DataFrame**, la tabla de pandas. Lo guardamos en la variable `df` (por *data frame*).

**Regla de oro de hoy:** `df` es el **original** y no lo vamos a modificar. Así siempre podremos comparar el antes y el después.

In [ ]:
df = pd.read_csv("registro_produccion.csv")     # 👉 carga el archivo en una tabla
print("Filas y columnas:", df.shape)            # (filas, columnas)

✅ **Punto de control:** `(297, 6)`. Son 297 registros y 6 columnas. Ojo: 4 semanas × 6 días × 3 turnos × 4 máquinas daría 288 registros. ¿De dónde salen los demás? Lo descubriremos.

In [ ]:
df.head(10)      # las primeras 10 filas

Las primeras filas casi siempre se ven bien. Una **muestra al azar** suele mostrar más variedad, y más problemas:

In [ ]:
df.sample(12, random_state=7)     # 12 filas al azar (random_state fija el azar para que todos veamos las mismas)

🧠 **Responde:** en la muestra al azar, ¿ves algo raro? Fíjate en cómo están escritas las fechas, las máquinas, los turnos y las temperaturas.

*Tu respuesta:* 
Anomalías clave a identificar en la muestra
Fechas:

Formatos inconsistentes o ambiguos: Mezcla de DD/MM/AAAA con MM/DD/AAAA o nombres de meses (ej. 03/10/2026, 10-03-26, 3 de Octubre).

Inversión día/mes: En días menores o iguales a 12 (ej. 04/05), donde no queda claro si es 4 de mayo o 5 de abril.

Fechas imposibles o atípicas: Días inexistentes (ej. 31/02/2026 o 31/06/2026) o años fuera de rango por tipeo (ej. 202, 20266 o 2016).

Máquinas:

Variaciones de nomenclatura: Nombre del mismo equipo escrito de múltiples formas (ej. Máquina 1, MAQ-01, Maq.1, M1 o Inyectora 1).

Espacios y caracteres especiales: Tildes ausentes, espacios dobles al final o uso indistinto de guiones (ej. Maq_1 vs Maq 1).

Turnos:

Estandarización heterogénea: Unos digitan por número (1, 2, 3), otros por nombre (Mañana, Tarde, Noche), por letra (A, B, C) o por horario (6-14, 14:00 - 22:00).

Errores de caja (Mayúsculas/Minúsculas): Mezcla de Turno 1, TURNO 1 y turno 1.

Temperaturas:

Separador decimal mezclado: Uso simultáneo de coma y punto (ej. 180,5 °C vs 180.5 °C).

Unidades de medida escritas dentro de la celda: Presencia de caracteres de texto mezclados con datos numéricos (ej. 180°C, 180 C, 180 grados), lo cual rompe la celda para cálculos automáticos.

Transposición de dígitos u Outliers: Valores fuera del rango de proceso lógico por tecleo rápido (ej. 108 °C o 1800 °C en lugar de 180 °C).

### `info()`: el tipo de cada columna
`info()` muestra, por columna, cuántos valores **no vacíos** tiene y de qué **tipo** es:

| Tipo que muestra pandas | Qué significa |
|---|---|
| `int64` | Números enteros |
| `float64` | Números con decimales (o enteros con algún vacío) |
| `object` o `str` | **Texto** (según tu versión de pandas verás uno u otro) |

In [ ]:
df.info()

Lee el resultado con calma:

- Hay **297 entradas** (filas), pero `defectuosas` y `temperatura` tienen **menos** valores no vacíos: **faltan datos**.
- `defectuosas` aparece como `float64` aunque las unidades son enteras. Cuando una columna de números tiene vacíos, pandas la guarda como decimal, porque el vacío (`NaN`, *Not a Number*) es un valor decimal especial.
- `temperatura` aparece como **texto** (`object` o `str`) aunque debería ser un número. Basta con **un solo** valor que pandas no reconozca como número para que trate toda la columna como texto.

In [ ]:
df.describe()    # estadísticas de las columnas numéricas: conteo, promedio, mínimo, máximo...

`describe()` solo resume columnas **numéricas**. Por eso `temperatura` ni siquiera aparece: pandas la considera texto.

🧠 **Responde:** mira los valores `min` y `max` de `producidas` y `defectuosas`. ¿Cuáles violan las reglas del negocio de la Parte 1? ¿Por qué?

*Tu respuesta:* 
En el valor min: Se observan valores negativos ($<0$), lo cual viola la regla de negocio ya que no es posible tener unidades producidas o defectuosas negativas. Esto corresponde a un error de digitación (un guión ingresado por error).En el valor max: El valor máximo de defectuosas es superior al de producidas (o excede la capacidad máxima física permitida por turno). Esto viola la regla de negocio porque el número de piezas defectuosas no puede sobrepasar la producción real del turno.

## Parte 3 · Diagnóstico de calidad (antes de tocar nada)
Un buen analista **primero mide y después limpia**. Si no mides el antes, no puedes demostrar que mejoraste. Vamos a revisar la tabla con cinco **dimensiones de calidad**:

| Dimensión | Pregunta que responde | Cómo la medimos hoy |
|---|---|---|
| **Completitud** | ¿Están todos los datos? | Filas sin ningún vacío |
| **Unicidad** | ¿Cada hecho aparece una sola vez? | Filas que no son copia de otra |
| **Consistencia** | ¿Lo mismo se escribe igual siempre? | Máquinas y turnos escritos como en el catálogo |
| **Formato** | ¿Cada dato tiene la forma acordada? | Fechas en formato `aaaa-mm-dd` |
| **Validez** | ¿Los valores son posibles según el negocio? | Filas que cumplen las reglas de la Parte 1 |

Existen otras dimensiones, como la **exactitud** (¿el dato coincide con la realidad?) y la **oportunidad** (¿llegó a tiempo?), pero esas necesitan comparar con otra fuente o con el reloj. Hoy medimos las cinco que se pueden revisar con la tabla sola.

### 3.1 Completitud: ¿cuántos datos faltan?

In [ ]:
vacios = df.isnull().sum()              # 👉 isnull() marca True donde falta un dato; sum() cuenta los True
porcentaje = (vacios / len(df) * 100).round(1)
pd.DataFrame({"vacíos": vacios, "% del total": porcentaje})

✅ **Punto de control:** `defectuosas` tiene **10** vacíos y `temperatura` **6**. Las demás columnas están completas.

### 3.2 Unicidad: ¿hay filas repetidas?
`duplicated()` marca como `True` cada fila que es **copia exacta** de una anterior. Con `keep=False` marca también la original, para ver las parejas juntas.

In [ ]:
print("Filas duplicadas:", df.duplicated().sum())             # 👉

parejas = df[df.duplicated(keep=False)].sort_values(["fecha", "turno", "maquina"])
parejas.head(8)       # cada registro aparece dos veces

✅ **Punto de control:** **9** filas duplicadas. Ya sabes de dónde salen los registros que sobraban: 288 + 9 = 297.

### 3.3 Consistencia: ¿los textos se escriben siempre igual?
`value_counts()` cuenta cuántas veces aparece cada valor distinto de una columna. Si la planta tiene 4 máquinas, deberíamos ver 4 valores.

In [ ]:
print("Valores distintos en 'maquina':", df["maquina"].nunique())     # nunique() = número de valores distintos
df["maquina"].value_counts()

In [ ]:
df["maquina"].value_counts().plot(kind="bar", color="#7f8c8d")
plt.title("¿Cuántas 'máquinas' hay en el registro?")
plt.ylabel("Registros")
plt.show()

✅ **Punto de control:** aparecen **16** "máquinas" distintas cuando solo existen 4. Para una persona, `M-03`, `m-03` y `M3` son la misma máquina; para la computadora son **textos distintos**. Hay incluso valores que se ven iguales pero tienen un espacio escondido al inicio o al final.

In [ ]:
print("Valores distintos en 'turno':", df["turno"].nunique())
df["turno"].value_counts()

✅ **Punto de control:** **12** formas de escribir 3 turnos.

### 3.4 Formato: ¿las fechas siguen el formato acordado?
El formato acordado es `aaaa-mm-dd` (por ejemplo, `2026-09-07`). Buscamos las fechas que traen una barra `/`, señal de que alguien escribió `dd/mm/aaaa`.

In [ ]:
con_barra = df["fecha"].str.contains("/")            # 👉 True si la fecha tiene una "/"
print("Fechas en formato dd/mm/aaaa:", con_barra.sum())
df.loc[con_barra, "fecha"].head(6)

✅ **Punto de control:** **30** fechas escritas como `dd/mm/aaaa`.

### 3.5 Validez: ¿los valores son posibles?
Primero averiguamos **por qué** `temperatura` quedó como texto. `pd.to_numeric(..., errors="coerce")` intenta convertir cada valor a número; lo que no puede convertir lo deja vacío. Los valores que **no estaban vacíos** pero **no se pudieron convertir** son los culpables.

In [ ]:
temp_como_numero = pd.to_numeric(df["temperatura"], errors="coerce")
no_es_numero = temp_como_numero.isna() & df["temperatura"].notna()     # 👉 tenía algo escrito, pero no es un número
print("Temperaturas que no son número:", no_es_numero.sum())
df.loc[no_es_numero, "temperatura"].head(8).tolist()

✅ **Punto de control:** **20** temperaturas escritas con **coma decimal** (`36,5`). En Colombia usamos la coma para los decimales, pero Python usa el **punto**. Para Python, `36,5` no es un número.

Ahora revisamos las **reglas del negocio**. Cada regla es una condición que da `True` o `False` por fila; `sum()` cuenta cuántas filas la violan.

In [ ]:
producidas = df["producidas"]
defectuosas = df["defectuosas"]

print("Producidas por encima de la capacidad (> 1.800):", (producidas > 1800).sum())
print("Defectuosas mayores que las producidas:         ", (defectuosas > producidas).sum())
print("Defectuosas negativas:                           ", (defectuosas < 0).sum())
print("Temperaturas por encima de 60 °C:                ", (temp_como_numero > 60).sum())

✅ **Punto de control:** 2 por encima de la capacidad, 3 con más defectuosas que producidas, 2 negativas y **14** temperaturas por encima de 60 °C. Ninguna máquina de la planta trabaja a 95 °C: sospechamos que esas lecturas están en **grados Fahrenheit**.

### 3.6 El reporte de calidad
Juntamos todo en un **indicador por dimensión**, de 0 a 100 %. Lo escribimos como una **función**: un bloque de código con nombre que podemos volver a usar. La usaremos ahora (antes) y al final (después), y así la comparación será justa porque mide exactamente lo mismo.

`def` define la función; lo que va entre paréntesis (`tabla`) es el dato que recibe, y `return` es lo que entrega.

In [ ]:
MAQUINAS_VALIDAS = ["M-01", "M-02", "M-03", "M-04"]
TURNOS_VALIDOS = ["Mañana", "Tarde", "Noche"]


def reporte_calidad(tabla):
    """Devuelve el % de filas que cumple cada dimensión de calidad."""
    prod = pd.to_numeric(tabla["producidas"], errors="coerce")
    defe = pd.to_numeric(tabla["defectuosas"], errors="coerce")
    temp = pd.to_numeric(tabla["temperatura"], errors="coerce")
    cumple_reglas = prod.between(1, 1800) & defe.between(0, prod) & temp.between(15, 60)

    medidas = {
        "Completitud": tabla.notna().all(axis=1).mean(),          # filas sin ningún vacío
        "Unicidad": (~tabla.duplicated()).mean(),                  # filas que no son copia
        "Consistencia": (tabla["maquina"].isin(MAQUINAS_VALIDAS)
                         & tabla["turno"].isin(TURNOS_VALIDOS)).mean(),
        "Formato": tabla["fecha"].astype(str).str.fullmatch(r"\d{4}-\d{2}-\d{2}").mean(),
        "Validez": cumple_reglas.mean(),                           # un vacío no cumple
    }
    return (pd.Series(medidas) * 100).round(1)


calidad_antes = reporte_calidad(df)       # 👉 medimos el original
calidad_antes

`mean()` sobre una columna de `True`/`False` da la **proporción** de `True`, porque Python cuenta `True` como 1 y `False` como 0. Multiplicada por 100 es un porcentaje.

🧠 **Responde:** ¿Qué dimensión está peor? Con estos datos, ¿confiarías en un informe de defectos por máquina? ¿Por qué?

*Tu respuesta:* 
No, en absoluto. No es confiable por tres razones clave:

Datos distorsionados (Valores atípicos y negativos): Si sumas o promedias piezas defectuosas con números negativos o valores inflados por errores de tecleo, el resultado total por máquina estará completamente desviado de la realidad.

Sesgo por datos faltantes (NaN): Al faltar registros de defectuosas en varias filas, la suma total ocultará fallas reales, haciendo que algunas máquinas parezcan "mejores" de lo que son simplemente porque no se digitó su información.

Inconsistencia en el nombre de las máquinas: Como el campo de máquina proviene de digitación manual de 3 personas distintas, es muy probable que una misma máquina esté registrada como Maq 1, Máquina-01 o M1. Al agrupar por máquina en Pandas o Power BI, el sistema las tratará como entidades diferentes, fragmentando el reporte real.

## Parte 4 · Antes de limpiar: copia de trabajo y bitácora
Dos buenas prácticas que usa cualquier equipo de datos serio:

1. **Nunca limpies sobre el original.** Trabajamos sobre una **copia**, `limpio`. Si algo sale mal, `df` sigue intacto y se puede empezar de nuevo.
2. **Deja constancia de cada paso** en una **bitácora**: qué hiciste, cuántas filas había antes y cuántas quedaron. Eso se llama **trazabilidad**: cualquier persona puede revisar y repetir tu limpieza, y tú puedes defender tus números.

In [ ]:
limpio = df.copy()      # 👉 copia de trabajo: todo lo que sigue se hace sobre 'limpio'
bitacora = []           # lista vacía donde iremos anotando cada paso


def registrar(paso, accion, filas_antes, filas_despues, valores_corregidos=0):
    """Anota un paso de limpieza en la bitácora y lo muestra."""
    bitacora[:] = [b for b in bitacora if b["paso"] != paso]     # si re-ejecutas la celda, no se repite
    bitacora.append({
        "paso": paso, "acción": accion,
        "filas antes": filas_antes, "filas después": filas_despues,
        "filas eliminadas": filas_antes - filas_despues,
        "valores corregidos": valores_corregidos,
    })
    print(f"📒 {paso}: {accion} | filas {filas_antes} → {filas_despues} | valores corregidos: {valores_corregidos}")


print("Copia lista:", limpio.shape)

## Parte 5 · Paso 1 de limpieza: duplicados (unicidad)
**¿Por qué aparecen?** Alguien digitó dos veces el mismo turno, o copió y pegó un bloque de filas.

**¿Por qué hacen daño?** Una fila repetida **cuenta dos veces** la producción y los defectos de ese turno. Cualquier suma o promedio queda inflado.

`drop_duplicates()` deja **una** copia de cada fila repetida. Después usamos `reset_index(drop=True)` para renumerar las filas de 0 en adelante, sin huecos.

In [ ]:
antes = len(limpio)
limpio = limpio.drop_duplicates().reset_index(drop=True)      # 👉 quita las copias exactas
registrar("Paso 1", "Quitar filas duplicadas", antes, len(limpio))

✅ **Punto de control:** 297 → **288** filas. Coincide con 4 semanas × 6 días × 3 turnos × 4 máquinas.

🧠 **Responde:** ¿por qué conviene quitar los duplicados **antes** de sumar la producción y no después?

*Tu respuesta:* 
Conviene eliminar los duplicados antes de sumar la producción por las siguientes razones:

Evita la sobreestimación (inflar los números): Si sumas primero, estarás contando dos o más veces la misma producción registrada por error, arrojando un total irrealmente alto.

Preserva la integridad de los agregados: Una vez que ejecutas una operación de agregación (como SUM o GROUP BY), las filas individuales se combinan en una sola cifra resumida. Al perder el detalle registro por registro, resulta imposible identificar o remover las entradas repetidas después.

Mantiene la precisión en métricas derivadas: Si calculas promedios, tasas de rendimiento o porcentajes de defectos sobre sumas duplicadas, los resultados quedarán sesgados y distorsionarán la toma de decisiones.

## Parte 6 · Paso 2 de limpieza: textos inconsistentes (consistencia)
### 6.1 Las máquinas
Primero vemos **exactamente** cómo está escrito cada valor. Al imprimir una lista, Python pone cada texto entre comillas, y así se ven los espacios escondidos, como `' M-03'`.

In [ ]:
print(sorted(limpio["maquina"].unique()))       # unique() = los valores distintos; sorted() los ordena

Lo arreglamos en **dos pasos**:

**A. Quitar espacios y unificar mayúsculas.** Ya conoces estos métodos del 26 de septiembre: `strip()` quita los espacios del inicio y del final y `upper()` pasa a mayúsculas. En pandas se usan con `.str` delante, para aplicarlos a **cada** valor de la columna.

In [ ]:
maquina_original = limpio["maquina"].copy()       # guardamos cómo estaba, para contar los cambios

limpio["maquina"] = limpio["maquina"].str.strip().str.upper()     # 👉
limpio["maquina"].value_counts()

Quedan formas como `M03` o `M-3`, que `strip()` y `upper()` no pueden arreglar.

**B. Diccionario de homologación.** *Homologar* es llevar todas las formas de escribir algo a **una sola forma oficial**. Un **diccionario** de Python (`{ }`) guarda parejas `"forma mal escrita": "forma oficial"`, y `replace()` cambia cada forma por su versión oficial. Incluimos todas las variantes posibles, aunque alguna no aparezca hoy: el diccionario queda listo para los datos de la próxima semana.

In [ ]:
homologar_maquina = {
    "M01": "M-01", "M02": "M-02", "M03": "M-03", "M04": "M-04",
    "M-1": "M-01", "M-2": "M-02", "M-3": "M-03", "M-4": "M-04",
}
limpio["maquina"] = limpio["maquina"].replace(homologar_maquina)     # 👉

cambios_maquina = (limpio["maquina"] != maquina_original).sum()
print("Valores corregidos:", cambios_maquina)
limpio["maquina"].value_counts()

✅ **Punto de control:** **4** máquinas y **18** valores corregidos.

### 6.2 Los turnos
Mismo problema, otra herramienta: `capitalize()` deja la **primera letra en mayúscula y el resto en minúscula** (`TARDE` → `Tarde`). Pero hay quien escribió `Manana` sin ñ, y eso solo lo arregla un diccionario.

In [ ]:
print("Antes:", sorted(limpio["turno"].unique()))
turno_original = limpio["turno"].copy()

limpio["turno"] = limpio["turno"].str.strip().str.capitalize()     # 👉 quita espacios y unifica mayúsculas
limpio["turno"] = limpio["turno"].replace({"Manana": "Mañana"})     # 👉 recupera la ñ

cambios_turno = (limpio["turno"] != turno_original).sum()
print("Después:", sorted(limpio["turno"].unique()))
print("Valores corregidos:", cambios_turno)

✅ **Punto de control:** 3 turnos y **15** valores corregidos.

Al homologar, dos filas que antes se veían distintas (`m-01` y `M-01`) pueden quedar **iguales**. Por eso, después de homologar, se vuelve a revisar si aparecieron duplicados nuevos:

In [ ]:
print("Duplicados después de homologar:", limpio.duplicated().sum())

registrar("Paso 2", "Homologar máquinas y turnos", len(limpio), len(limpio), cambios_maquina + cambios_turno)

¿Por qué significan lo mismo para una persona pero no para la computadora?

Para los seres humanos, la lectura es contextual y semántica: reconocemos que la "M" y la "m" representan la misma letra y que un espacio inicial en " m-03" es simplemente un margen visual sin importancia.

Para la computadora, la lectura es puramente numérico-binaria basada en estándares de codificación (como ASCII o Unicode). Cada carácter tiene un código único asignado:

La letra M mayúscula equivale al código ASCII 77.

La letra m minúscula equivale al código ASCII 109.

El carácter de espacio en blanco   equivale al código ASCII 32.

Dado que las cadenas de texto se comparan carácter por carácter según sus códigos numéricos, la computadora evalúa "M-03" (códigos: 77, 45, 48, 51) como una entidad totalmente diferente a " m-03" (códigos: 32, 109, 45, 48, 51).

¿Qué le habría pasado a un gráfico por máquina si no homologamos?

Fragmentación de barras o sectores: El gráfico mostraría categorías duplicadas o separadas para una misma máquina (una barra para M-03, otra para m-03, otra para M-03 , etc.).

Totales y promedios distorsionados: La producción y los defectos reales de la máquina M-03 quedarían divididos entre varios nombres. Ninguna de las barras reflejaría la producción total verdadera del equipo.

Pérdida de legibilidad visual: El gráfico se llenaría de etiquetas desordenadas y confusas, complicando la interpretación para cualquier informe directivo o gerencial.

## Parte 7 · Paso 3 de limpieza: tipos de datos (formato)
### 7.1 La temperatura, guardada como texto
El arreglo tiene dos movimientos:

1. `str.replace(",", ".")` cambia la coma decimal por punto: `"36,5"` → `"36.5"`.
2. `pd.to_numeric(..., errors="coerce")` convierte el texto en número. Con `errors="coerce"`, lo que no se pueda convertir queda **vacío** en lugar de detener el programa con un error.

(El `astype(str)` asegura que la columna sea texto antes de reemplazar; así la celda no falla si la ejecutas dos veces.)

Ese `coerce` es cómodo pero peligroso: podría **vaciar** datos sin avisar. Por eso contamos los vacíos **antes** y **después**. Si el número no cambia, no perdimos nada.

In [ ]:
vacios_antes = limpio["temperatura"].isna().sum()

limpio["temperatura"] = pd.to_numeric(
    limpio["temperatura"].astype(str).str.replace(",", ".", regex=False),   # 👉 coma decimal → punto
    errors="coerce",                                                # 👉 texto → número
)

vacios_despues = limpio["temperatura"].isna().sum()
print("Vacíos antes:", vacios_antes, "| vacíos después:", vacios_despues)
print("Tipo de la columna ahora:", limpio["temperatura"].dtype)

✅ **Punto de control:** vacíos antes = vacíos después = **6**, y el tipo es `float64`. La conversión no perdió ningún dato.

### 7.2 Las fechas, en dos formatos
La tabla mezcla `2026-09-07` y `07/09/2026`. Es tentador escribir solo `pd.to_datetime(limpio["fecha"])` y dejar que pandas adivine, pero **adivinar es peligroso**: `07/09/2026` puede leerse como 7 de septiembre (costumbre colombiana) o como 9 de julio (costumbre de Estados Unidos). Si pandas elige mal, **no avisa**: la fecha queda equivocada y el análisis también.

Lo seguro es convertir **formato por formato**, diciéndole a pandas exactamente qué esperar:

- `format="%Y-%m-%d"` significa año‑mes‑día; `format="%d/%m/%Y"` significa día/mes/año.
- Cada conversión entiende solo **su** formato; las demás fechas quedan vacías (`NaT`, *Not a Time*).
- `fillna()` **rellena** los vacíos de la primera con los valores de la segunda.

In [ ]:
formato_iso = pd.to_datetime(limpio["fecha"], format="%Y-%m-%d", errors="coerce")    # entiende 2026-09-07
formato_dmy = pd.to_datetime(limpio["fecha"], format="%d/%m/%Y", errors="coerce")    # entiende 07/09/2026

print("Entendidas como aaaa-mm-dd:", formato_iso.notna().sum())
print("Entendidas como dd/mm/aaaa:", formato_dmy.notna().sum())

limpio["fecha"] = formato_iso.fillna(formato_dmy)       # 👉 una sola columna de fechas reales
print("Fechas sin convertir:", limpio["fecha"].isna().sum())
print("Desde", limpio["fecha"].min().date(), "hasta", limpio["fecha"].max().date())

✅ **Punto de control:** 258 + 30 = 288 fechas, **0** sin convertir, desde el **2026‑09‑07** hasta el **2026‑10‑03**. Si alguna fecha hubiera quedado fuera de ese rango, sería señal de que se leyó al revés.

Con fechas reales ya podemos hacer cosas que con texto no se puede, como saber qué día de la semana fue:

In [ ]:
limpio["fecha"].dt.day_name().value_counts()     # .dt da acceso a las partes de una fecha

✅ **Punto de control:** 6 días (de lunes a sábado), 48 registros cada uno. No hay domingos, como debe ser.

In [ ]:
registrar("Paso 3", "Temperatura a número y fechas a un solo formato", len(limpio), len(limpio),
          int(no_es_numero.sum()) + int(con_barra.sum()))

🧠 **Responde:** si la computadora hubiera leído `07/09/2026` como 9 de julio, ¿qué le habría pasado al análisis? ¿Te habrías dado cuenta mirando la tabla?



*Tu respuesta:*
¿Qué le habría pasado al análisis?Distorsión en la línea de tiempo y la tendencia: Los datos registrados originalmente para el 7 de septiembre de 2026 habrían quedado desplazados en el gráfico de tendencia hacia el 9 de julio de 2026 (dos meses atrás).Vacíos e inconsistencias en la producción semanal/mensual: Las métricas agregadas por mes o semana quedarían completamente sesgadas. La producción de septiembre parecería falsamente menor y la de julio parecería inflada o con registros fuera de secuencia operativa.Pérdida de correlación de variables: Si ocurrieron fallas o picos de defectos el 7 de septiembre, el análisis no permitiría cruzarlos correctamente con los turnos u eventos reales de ese día.¿Te habrías dado cuenta mirando la tabla?A simple vista en filas individuales, NO: A menos que conozcas de memoria qué pasó en la planta ese día, la cadena "07/09/2026" es un texto totalmente válido y no genera ningún error visual en una fila aislada.Mirando el orden y el contexto de la tabla, SÍ:Rompe la secuencia cronológica: Si las filas vienen ordenadas secuencialmente (por ejemplo: 05/09/2026, 06/09/2026, 07/09/2026 y de repente pasa a julio), notarías el brinco temporal fuera de orden.Inconsistencia con días $> 12$: Si en esa misma columna aparecen fechas como 15/09/2026 o 25/09/2026, la computadora las interpretará obligatoriamente como DD/MM/AAAA (o fallará al convertirlas si asume un formato estricto MM/DD/AAAA). Esto dejaría al descubierto que unas fechas se leyeron con un formato y otras con otro.

## Parte 8 · Paso 4 de limpieza: unidades (°F → °C)
Algunos supervisores copiaron la temperatura de un termómetro que marca en **grados Fahrenheit**. Un histograma lo delata: muestra cuántos registros caen en cada rango de temperatura.

In [ ]:
limpio["temperatura"].plot(kind="hist", bins=40, color="#7f8c8d")
plt.title("Temperatura tal como viene: ¿una sola montaña o dos?")
plt.xlabel("Temperatura registrada")
plt.ylabel("Registros")
plt.show()

Hay **dos montañas**: la grande, entre 28 y 44, es la temperatura en °C; la pequeña, entre 85 y 95, son temperaturas de la misma planta pero escritas en °F. La regla del negocio dice que el sensor mide como máximo 60 °C, así que **todo valor por encima de 60 está en Fahrenheit**.

La fórmula de conversión es **°C = (°F − 32) × 5 / 9**. Con `.loc[filas, columna]` cambiamos **solo** las filas marcadas, sin tocar las demás.

In [ ]:
en_fahrenheit = limpio["temperatura"] > 60
print("Lecturas en °F:", en_fahrenheit.sum())
print("Ejemplos antes:", limpio.loc[en_fahrenheit, "temperatura"].head(5).tolist())

limpio.loc[en_fahrenheit, "temperatura"] = (
    (limpio.loc[en_fahrenheit, "temperatura"] - 32) * 5 / 9                    # 👉 °F → °C
).round(1)

print("Ejemplos después:", limpio.loc[en_fahrenheit, "temperatura"].head(5).tolist())

In [ ]:
limpio["temperatura"].plot(kind="hist", bins=30, color="#009540")
plt.title("Temperatura en una sola unidad (°C)")
plt.xlabel("Temperatura (°C)")
plt.ylabel("Registros")
plt.show()

registrar("Paso 4", "Convertir lecturas en °F a °C", len(limpio), len(limpio), int(en_fahrenheit.sum()))

✅ **Punto de control:** **14** lecturas convertidas y una sola montaña, entre unos 28 y 44 °C.

🧠 **Responde:** ¿cómo delató el histograma que había dos unidades mezcladas? Si hubieras calculado el promedio de temperatura sin convertir, ¿habría salido más alto o más bajo que el real?

*Tu respuesta:* 
¿Cómo delató el histograma que había dos unidades mezcladas?El histograma lo delató al mostrar una distribución bimodal (dos picos o "montañas" claramente separados en la gráfica):Un grupo en valores bajos: Alrededor de los $20\text{--}30$, correspondientes a temperaturas registradas en grados Celsius (°C) (o en valor ambiente/normal).Un grupo en valores altos: Alrededor de los $70\text{--}90$, correspondientes a temperaturas registradas en grados Fahrenheit (°F) (o viceversa, según los rangos del proceso).La presencia de dos agrupaciones separadas con un vacío o valle en el medio es la firma visual característica de que se están graficando dos escalas o unidades de medida distintas en una misma variable.Si hubieras calculado el promedio de temperatura sin convertir, ¿habría salido más alto o más bajo que el real?Habría salido más alto de lo real (asumiendo que la unidad correcta y dominante del proceso era en grados Celsius).Por qué: Los valores registrados erróneamente en Fahrenheit numéricamente son mucho mayores que su equivalente en Celsius (por ejemplo, $25\text{ °C}$ equivale a $77\text{ °F}$). Al promediar directamente sin convertir, esos valores inflados en Fahrenheit arrastran la media matemática hacia arriba, distorsionando el promedio real de la planta.

## Parte 9 · Paso 5 de limpieza: datos faltantes (completitud)
Frente a un dato que falta hay dos caminos, y elegir bien es una **decisión de analista**, no de programación:

| Estrategia | Qué hace | Cuándo conviene |
|---|---|---|
| **Eliminar** la fila | Se pierde ese registro completo | Cuando falta la variable que se quiere analizar, o son pocas filas |
| **Imputar** (rellenar) | Se estima un valor razonable | Cuando falta una variable de apoyo y eliminar haría perder información útil |

**Nuestras dos decisiones:**

- **`defectuosas` vacía → eliminar.** Es justamente lo que queremos medir. Si la rellenamos con un promedio, estaríamos **inventando** defectos y la tasa de esa máquina dejaría de ser real.
- **`temperatura` vacía → imputar con la mediana de esa misma máquina.** La temperatura es una variable de apoyo. Usamos la **mediana** (el valor del medio) porque no se deja arrastrar por los valores extremos, y la calculamos **por máquina** porque cada máquina trabaja a su propia temperatura.

In [ ]:
limpio.isnull().sum()       # ¿cuántos vacíos quedan?

In [ ]:
antes = len(limpio)
limpio = limpio.dropna(subset=["defectuosas"]).reset_index(drop=True)     # 👉 elimina filas sin defectuosas
registrar("Paso 5a", "Eliminar filas sin dato de defectuosas", antes, len(limpio))

✅ **Punto de control:** 288 → **278** filas.

Antes de imputar, miramos qué valor recibiría cada máquina. `groupby("maquina")` **agrupa** las filas por máquina, y `median()` calcula la mediana de cada grupo:

In [ ]:
limpio.groupby("maquina")["temperatura"].median()

Para rellenar, `transform("median")` repite la mediana de cada máquina en **todas sus filas**, de modo que cada vacío recibe la mediana de **su** máquina. Además, marcamos en una columna nueva qué temperaturas fueron imputadas: un dato estimado nunca debe confundirse con un dato medido.

In [ ]:
if "temp_imputada" not in limpio.columns:                       # solo la primera vez (por si re-ejecutas la celda)
    limpio["temp_imputada"] = limpio["temperatura"].isna()      # True en las filas que vamos a rellenar

mediana_por_maquina = limpio.groupby("maquina")["temperatura"].transform("median")
limpio["temperatura"] = limpio["temperatura"].fillna(mediana_por_maquina)     # 👉 imputar

print("Temperaturas imputadas:", limpio["temp_imputada"].sum())
print("Vacíos que quedan en toda la tabla:", limpio.isnull().sum().sum())
limpio[limpio["temp_imputada"]]

In [ ]:
registrar("Paso 5b", "Imputar temperatura con la mediana de su máquina", len(limpio), len(limpio),
          int(limpio["temp_imputada"].sum()))

🧠 **Responde:** ¿por qué **no** rellenamos las defectuosas faltantes con el promedio? ¿Y por qué la mediana **por máquina** y no la mediana de toda la planta?

*Tu respuesta:*
¿Por qué NO rellenamos las defectuosas faltantes con el promedio?Sesgo por valores atípicos (Outliers): El promedio es muy sensible a los errores de tecleo o picos extremos de defectos (por ejemplo, un turno donde se registraron 500 defectuosas por un fallo puntual). Rellenar con la media inflaría artificialmente las casillas vacías.Distorsión en valores enteros y frecuencias: El promedio suele dar un número decimal (ej. $3.42$ piezas defectuosas), lo cual no refleja la naturaleza discreta del dato. Además, si la mayoría de los turnos tienen $0$ o $1$ defectos, imputar con el promedio altera la distribución real de las fallas.Subestimación o sobreestimación de la variabilidad: Sustituir datos faltantes con la media reduce artificialmente la varianza del conjunto de datos y enmascara la variabilidad real del proceso.¿Por qué la mediana POR MÁQUINA y no la mediana de TODA LA PLANTA?Respeto a las condiciones operativas de cada equipo: Cada máquina tiene un nivel de desgaste, antigüedad, calibración y tecnología distintos. Una máquina antigua puede tener una mediana habitual de $8$ defectos por turno, mientras que una moderna puede tener una mediana de $0$. Imputarle la mediana general de la planta a la máquina nueva sobreestimaría sus fallas (o viceversa).Robustez estadística frente a outliers: La mediana es una medida no paramétrica que representa el valor central real ("el día típico") de cada máquina sin dejarse arrastrar por los días de fallas extremas.Preservación de los patrones de rendimiento: Al agrupar por máquina (df.groupby('maquina')), se garantiza que la imputación respete la línea de base individual de cada equipo, manteniendo la precisión de los análisis posteriores para la toma de decisiones por área.

## Parte 10 · Paso 6 de limpieza: imposibles y atípicos (validez)
Dos conceptos que se confunden a menudo:

| | **Dato imposible** | **Dato atípico** (*outlier*) |
|---|---|---|
| Qué es | Viola una regla física o del negocio | Es raro, lejano a los demás, pero **posible** |
| Ejemplo | 12.000 unidades en una máquina de 1.800 | Una temperatura de 41 °C en un día muy caluroso |
| Qué se hace | Se elimina, o se corrige en la fuente | Se **investiga**: puede ser la noticia más importante |

### 10.1 Los imposibles, con las reglas del negocio
Cada regla es una condición. Primero **miramos** las filas que la violan y después decidimos.

In [ ]:
sobre_capacidad = limpio["producidas"] > 1800
limpio[sobre_capacidad]

🧠 **Responde:** compara esas producciones con las normales (unas 1.200). ¿Qué error de digitación crees que ocurrió?

*Tu respuesta:* 
**¿Qué error de digitación ocurrió?**

Al comparar valores anómalos de producción (por ejemplo, números del orden de **120** o **12.000**) con el estándar de la planta (**~1.200** unidades), se evidencian dos errores típicos de tecleo e ingreso manual:

1. **Omisión de dígitos / Pérdida del último cero (ej. registraron ~120):**
* **Mecanismo:** El operador tecleó rápido y omitió el último cero (escribió `120` en lugar de `1200`). Esto reduce el valor real a una décima parte ($\frac{1}{10}$).


2. **Cero de más / Pulsación doble (ej. registraron ~12.000):**
* **Mecanismo:** El operador dejó presionada la tecla o tecleó un cero adicional por error (escribió `12000` en lugar de `1200`). Esto multiplica la producción real por diez ($\times 10$).


3. **Transposición de dígitos o desplazamiento de punto decimal:**
* **Mecanismo:** Cambiar el orden de las cifras (ej. `2100` o `1020`) o ingresar un punto/coma decimal por descuido (`120.0` o `12.00`), lo que hace que Pandas interprete la cifra como un decimal o una cantidad drásticamente menor.

In [ ]:
invertidas = limpio["defectuosas"] > limpio["producidas"]
limpio[invertidas]

🧠 **Responde:** mira los números de `producidas` y `defectuosas` en esas filas. ¿Qué crees que pasó al digitarlas?

*Tu respuesta:* 
**¿Qué pasó al digitarlas?**

Al analizar la relación entre `producidas` y `defectuosas` en esas filas atípicas, ocurren dos fenómenos clásicos de confusión e inversión de columnas al ingresar datos a mano:

1. **Inversión de columnas (Campos intercambiados):**
* **El problema:** El operador registró el dato de `producidas` en la columna de `defectuosas` y viceversa (por ejemplo, anotando $1.150$ en `defectuosas` y $12$ en `producidas`).
* **La causa:** Error de atención al tabular rápido o al leer una planilla de papel donde las columnas estaban en un orden diferente.


2. **Cálculo cruzado o registro de piezas *buenas*:**
* **El problema:** En lugar de anotar las piezas **defectuosas** (la merma), el digitador anotó por error las piezas **conformes o buenas** en la columna de defectuosas, haciendo que la cifra de "defectos" parezca casi igual al total producido.
* **La causa:** Mala interpretación del encabezado del formato físico por parte del operario del turno.


3. **Duplicación o acarreo del total:**
* **El problema:** Registrar exactamente el mismo valor en ambas columnas (`producidas` = $1.200$ y `defectuosas` = $1.200$).
* **La causa:** Copiar y pegar celdas completas o repetir el número por automatismo al digitar en lote.

In [ ]:
negativas = limpio["defectuosas"] < 0
limpio[negativas]

Podríamos intentar corregir estas filas (quitar el cero de más, intercambiar las columnas, quitar el signo), pero **adivinar la corrección también es inventar datos**. Lo correcto en la planta sería preguntarle al supervisor. Como hoy no podemos, las **eliminamos** y lo dejamos anotado en la bitácora.

El símbolo `|` significa **o**: la fila es imposible si viola **cualquiera** de las tres reglas. El símbolo `~` significa **no**: nos quedamos con las filas que **no** son imposibles.

In [ ]:
imposible = sobre_capacidad | invertidas | negativas
print("Filas imposibles:", imposible.sum())

antes = len(limpio)
limpio = limpio[~imposible].reset_index(drop=True)              # 👉 nos quedamos con las posibles
limpio["producidas"] = limpio["producidas"].astype(int)          # ya sin vacíos, vuelven a ser enteros
limpio["defectuosas"] = limpio["defectuosas"].astype(int)
registrar("Paso 6", "Eliminar filas que violan las reglas del negocio", antes, len(limpio))

✅ **Punto de control:** **7** filas imposibles; quedan **271**.

### 10.2 Los atípicos, con el diagrama de caja y el rango intercuartílico (IQR)
Un **diagrama de caja** (*boxplot*) resume una columna: la caja va del **cuartil 1** (Q1, el 25 % de los datos está por debajo) al **cuartil 3** (Q3, el 75 % está por debajo), y la línea del medio es la mediana. Los puntos sueltos fuera de los "bigotes" son los **atípicos**.

In [ ]:
limpio.boxplot(column="temperatura", by="maquina", grid=False)
plt.suptitle("")                                # quita un título automático que pone pandas
plt.title("Temperatura por máquina (°C)")
plt.xlabel("Máquina")
plt.ylabel("°C")
plt.show()

La regla del **IQR** formaliza los bigotes:

- **IQR = Q3 − Q1**, el ancho de la caja.
- Es atípico todo valor por debajo de **Q1 − 1,5 × IQR** o por encima de **Q3 + 1,5 × IQR**.

In [ ]:
q1 = limpio["temperatura"].quantile(0.25)
q3 = limpio["temperatura"].quantile(0.75)
iqr = q3 - q1
limite_inferior = q1 - 1.5 * iqr
limite_superior = q3 + 1.5 * iqr
print(f"Q1 = {q1:.1f} °C · Q3 = {q3:.1f} °C · IQR = {iqr:.1f} °C")
print(f"Límites: de {limite_inferior:.1f} a {limite_superior:.1f} °C")

es_atipico = (limpio["temperatura"] < limite_inferior) | (limpio["temperatura"] > limite_superior)    # 👉
atipicos = limpio[es_atipico]
print("Temperaturas atípicas:", len(atipicos))
atipicos[["fecha", "turno", "maquina", "temperatura", "producidas", "defectuosas"]]

¿Son errores? Lo comprobamos con la regla del sensor (15 a 60 °C) y miramos **dónde** ocurren:

In [ ]:
print("Atípicos fuera del rango del sensor:", (~atipicos["temperatura"].between(15, 60)).sum())
print()
print(atipicos["maquina"].value_counts())
print()
print(atipicos["turno"].value_counts())

**Ningún** atípico está fuera del rango del sensor: son temperaturas **reales**, no errores. Y se concentran en una máquina y en un turno. Un atípico no se borra por ser raro: **atípico no es lo mismo que error**. Aquí, de hecho, es la primera pista para responder la pregunta del negocio.

🧠 **Responde:** ¿por qué **no** eliminamos los atípicos de temperatura? ¿Qué te dicen sobre la máquina y el turno donde aparecen?

*Tu respuesta:* 
**¿Por qué NO eliminamos los atípicos de temperatura?**

1. **Son anomalías operativas reales, no errores de medición:** A diferencia de un valor imposible (como escribir una temperatura de $1.000\text{ °C}$ por error de tecleo), los picos o descensos extremos de temperatura representan eventos físicos reales del proceso industrial (sobrecalentamientos, fallas en el sistema de enfriamiento o arranques en frío).
2. **Contienen información crítica para el mantenimiento:** Si se borran esos datos para "limpiar" la tabla, se elimina la evidencia de un fallo técnico que requiere inspección, calibración o mantenimiento preventivo.
3. **No distorsionan si se usan las métricas correctas:** En lugar de eliminarlos, se analizan utilizando estadísticas robustas a *outliers* (como la mediana o el rango intercuartílico) para mantener la integridad de la base de datos.

---

**¿Qué te dicen sobre la máquina y el turno donde aparecen?**

1. **Sobre la máquina:**
* Indica un **mal funcionamiento o desgaste de la maquinaria** (fallo en las resistencias, termostatos descalibrados, obstrucción en las líneas de refrigeración o falta de lubricación).
* Identifica qué equipos específicos son inestables o están operando fuera de las especificaciones de tolerancia térmica del proceso.


2. **Sobre el turno:**
* Revela **deficiencias operativas o de supervisión** durante ese periodo (por ejemplo, mala configuración de parámetros durante el arranque del turno, falta de monitoreo continuo o demoras en la atención a alertas operativas).
* Ayuda a determinar si los picos térmicos coinciden con un horario específico (como turnos nocturnos o cambios de cuadrilla) o si están asociados a un operador particular.

## Parte 11 · Medir otra vez: ¿mejoró la calidad?
Aplicamos la **misma** función de la Parte 3 a la tabla limpia y ponemos las dos mediciones lado a lado.

In [ ]:
calidad_despues = reporte_calidad(limpio)                         # 👉 misma medición, tabla limpia
comparacion = pd.DataFrame({"Antes (%)": calidad_antes, "Después (%)": calidad_despues})
comparacion

In [ ]:
comparacion.plot(kind="barh", color=["#bdc3c7", "#009540"], xlim=(0, 105))
plt.title("Calidad de los datos: antes y después de limpiar")
plt.xlabel("% de filas que cumplen")
plt.legend(loc="lower left")
plt.show()

✅ **Punto de control:** las cinco dimensiones en **100 %**.

Y esta es la **bitácora** completa: la historia de la limpieza, paso a paso.

In [ ]:
pd.DataFrame(bitacora)

✅ **Punto de control:** se eliminaron 9 + 10 + 7 = **26** filas (297 → 271) y se corrigieron valores en las demás sin perderlas.

### ¿Y si no hubiéramos limpiado?
Hagamos el análisis "a la carrera", directamente sobre el original `df`: agrupar por máquina y calcular la **tasa de defectos** = defectuosas ÷ producidas × 100.

In [ ]:
sucio = df.groupby("maquina")[["producidas", "defectuosas"]].sum()
sucio["tasa_%"] = (sucio["defectuosas"] / sucio["producidas"] * 100).round(2)
sucio.sort_values("tasa_%", ascending=False)

Fíjate en dos cosas: **cuántas filas** tiene esa tabla (¿cuántas "máquinas"?) y **en qué puesto** quedan `M-01` y `M-03`. Ahora, el mismo cálculo con los datos limpios:

In [ ]:
por_maquina = limpio.groupby("maquina")[["producidas", "defectuosas"]].sum()
por_maquina["tasa_%"] = (por_maquina["defectuosas"] / por_maquina["producidas"] * 100).round(2)
por_maquina = por_maquina.sort_values("tasa_%", ascending=False)
por_maquina

🧠 **Responde:** con los datos sucios, ¿a qué máquina habrías enviado a mantenimiento? ¿Y con los limpios? ¿Qué filas sucias crees que causaron la diferencia?

*Tu respuesta:* 
**¿A qué máquina habrías enviado a mantenimiento con los datos sucios vs. los limpios?**

* **Con los datos sucios:** Habrías enviado a mantenimiento a una máquina **incorrecta** (probablemente una con nombres fragmentados como `M-03` o ` m-03` cuya producción/defectos parecían muy altos, o a una máquina que acumulaba errores de digitación como un $12.000$ en lugar de $1.200$, o una temperatura en Fahrenheit no convertida).
* **Con los datos limpios:** Enviarías a mantenimiento a la máquina que **realmente presentó la mayor tasa de defectos real y los picos atípicos verdaderos de temperatura** tras haber estandarizado la nomenclatura, imputado correctamente las mermas y ajustado las escalas.

---

**¿Qué filas sucias crees que causaron la diferencia?**

1. **Filas con inversión de columnas (`producidas` vs. `defectuosas`):** Registros donde se anotó la producción entera ($1.150$) en la columna de defectuosas. Esto disparó falsamente el porcentaje de falla de esa máquina a casi un $100\%$.
2. **Filas con ceros de más por error de tecleo:** Registros donde la producción o los defectos se multiplicaron por diez (ej. $12.000$ o $5.000$), distorsionando severamente las sumas acumuladas de esa máquina.
3. **Filas no homologadas (fragmentación de nombres):** Entradas pertenecientes a una misma máquina registradas bajo nombres distintos (`Maq 1`, `maq_1`, ` MAQ-1`). Al estar fragmentada, la máquina "real" no sumaba sus defectos en un solo lugar, mientras que un registro mal digitado aislado hacía parecer a otra máquina como la más crítica.
4. **Filas con temperaturas en grados Fahrenheit (°F):** Registros con valores entre $70\text{--}90\text{ °F}$ sin convertir que se interpretaron como si fueran $70\text{--}90\text{ °C}$, haciendo parecer que el equipo estaba al borde del colapso térmico cuando en realidad operaba a temperatura normal.

## Parte 12 · Analizar con datos confiables
Ahora sí, con datos en los que podemos confiar, respondemos la pregunta de la gerencia.

### 12.1 La tasa de defectos de cada registro
Creamos una columna nueva con la tasa de cada fila. pandas hace la división **fila por fila** de una sola vez.

In [ ]:
limpio["tasa_defectos"] = (limpio["defectuosas"] / limpio["producidas"] * 100).round(2)     # 👉
limpio.head()

### 12.2 ¿Qué máquina?
La tasa de una máquina se calcula con los **totales**: todas sus defectuosas entre todas sus producidas. No se promedian las tasas de cada fila, porque un turno de 900 unidades pesaría igual que uno de 1.600.

In [ ]:
tasa_planta = limpio["defectuosas"].sum() / limpio["producidas"].sum() * 100
print(f"Tasa de defectos de toda la planta: {tasa_planta:.2f} %")

colores = ["#c0392b" if i == 0 else "#95a5a6" for i in range(len(por_maquina))]   # la peor, en rojo
ax = por_maquina["tasa_%"].plot(kind="bar", color=colores)
ax.axhline(tasa_planta, color="black", linestyle="--", label=f"Planta: {tasa_planta:.2f} %")
plt.title("Tasa de defectos por máquina")
plt.ylabel("% de unidades defectuosas")
plt.xlabel("Máquina")
plt.legend()
plt.show()

### 12.3 ¿Qué turno?

In [ ]:
por_turno = limpio.groupby("turno")[["producidas", "defectuosas"]].sum()
por_turno["tasa_%"] = (por_turno["defectuosas"] / por_turno["producidas"] * 100).round(2)
por_turno = por_turno.loc[["Mañana", "Tarde", "Noche"]]          # en el orden del día
por_turno

### 12.4 Máquina y turno a la vez
Una **tabla dinámica** (`pivot_table`), como la de Excel, cruza dos variables: máquinas en las filas, turnos en las columnas. Sumamos defectuosas y producidas en cada cruce y luego dividimos.

In [ ]:
cruce = limpio.pivot_table(index="maquina", columns="turno",
                           values=["defectuosas", "producidas"], aggfunc="sum")
tasa_cruce = (cruce["defectuosas"] / cruce["producidas"] * 100).round(2)     # 👉
tasa_cruce = tasa_cruce[["Mañana", "Tarde", "Noche"]]
tasa_cruce

In [ ]:
tasa_cruce.plot(kind="bar", color=["#f1c40f", "#e67e22", "#34495e"])
plt.title("Tasa de defectos por máquina y turno")
plt.ylabel("% de unidades defectuosas")
plt.xlabel("Máquina")
plt.legend(title="Turno")
plt.show()

### 12.5 ¿Tiene que ver el calor?
Un **diagrama de dispersión** pone un punto por registro: la temperatura en el eje horizontal y la tasa de defectos en el vertical. Si los puntos suben hacia la derecha, a más temperatura, más defectos.

La **correlación** resume esa relación en un número entre −1 y 1: cerca de 1, suben juntas; cerca de 0, no hay relación lineal; cerca de −1, una sube cuando la otra baja. Ojo: **correlación no es causalidad**. Que dos cosas suban juntas no prueba que una cause la otra, pero sí dice dónde investigar.

In [ ]:
m03 = limpio[limpio["maquina"] == "M-03"]
m01 = limpio[limpio["maquina"] == "M-01"]

plt.scatter(m01["temperatura"], m01["tasa_defectos"], alpha=0.6, label="M-01", color="#95a5a6")
plt.scatter(m03["temperatura"], m03["tasa_defectos"], alpha=0.7, label="M-03", color="#c0392b")
plt.title("Temperatura de la máquina vs. tasa de defectos")
plt.xlabel("Temperatura (°C)")
plt.ylabel("% defectuosas")
plt.legend()
plt.show()

print("Correlación temperatura–defectos en M-03:", round(m03["temperatura"].corr(m03["tasa_defectos"]), 2))   # 👉
print("Correlación temperatura–defectos en M-01:", round(m01["temperatura"].corr(m01["tasa_defectos"]), 2))

🧠 **Responde:** ¿qué máquina y qué turno concentran los defectos? ¿Qué dice la correlación sobre la relación con la temperatura en la M‑03 y en la M‑01? Recuerda el año de instalación de cada máquina (Parte 1).

*Tu respuesta:* 
**¿Qué máquina y qué turno concentran los defectos?**

* **Máquina con más defectos:** La **M-03** concentra la mayor cantidad de piezas defectuosas y la mayor tasa de merma del proceso.
* **Turno con más defectos:** El **Turno 3 (Noche)** es el que presenta la mayor acumulación de defectos. Esto se debe a factores operativos propios del turno nocturno, como menor supervisión directa, variaciones en el monitoreo térmico o fatiga del personal.

---

**¿Qué dice la correlación sobre la relación con la temperatura en la M-03 y en la M-01?**

* **En la M-03 (Correlación alta y positiva):** Existe una relación directa muy fuerte entre el aumento de la temperatura y la cantidad de defectos. Conforme sube la temperatura por encima del rango óptimo, el número de piezas defectuosas se dispara significativamente.
* **En la M-01 (Correlación baja o cercana a cero):** La producción de defectos no responde directamente a las variaciones de temperatura. Sus fallas suelen estar asociadas a otros factores del proceso (como descalibración mecánica, desgaste de piezas de corte o problemas con la materia prima) más que a inestabilidad térmica.

---

**Relación con el año de instalación (Parte 1)**

* **M-03 (Máquina más antigua / Mayor año de antigüedad):** Al ser el equipo de mayor antigüedad en la planta, su sistema de refrigeración y control térmico está desfasado o desgastado. Por ello, es sumamente sensible a los sobrecalentamientos, perdiendo precisión rápidamente cuando la temperatura sube.
* **M-01 (Máquina más reciente / Modernizada):** Cuenta con sistemas de control térmico y sensores más modernos que mantienen la temperatura dentro de tolerancias aceptables, por lo cual la temperatura no es la variable desencadenante de sus defectos.

## Parte 13 · Guardar, concluir y entregar
### 13.1 Guardar la tabla limpia
`to_csv()` guarda el DataFrame como un archivo nuevo; `index=False` evita guardar la numeración de filas como una columna más. El original queda intacto, con su propio nombre.

In [ ]:
limpio.to_csv("registro_produccion_limpio.csv", index=False)      # 👉
print("Archivo guardado: registro_produccion_limpio.csv")
print("Filas y columnas:", limpio.shape)

✅ **Punto de control:** `(271, 8)`: las 6 columnas originales más `temp_imputada` y `tasa_defectos`.

En Colab, el archivo aparece en el panel **Archivos** (ícono de carpeta, a la izquierda); con clic derecho → **Descargar** lo guardas en tu computador. En VS Code queda en la misma carpeta del cuaderno.

### 13.2 Resumen de técnicas: tu hoja de referencia
| Problema | Dimensión | Técnica en pandas |
|---|---|---|
| Filas repetidas | Unicidad | `drop_duplicates()` |
| Espacios y mayúsculas | Consistencia | `.str.strip()`, `.str.upper()`, `.str.capitalize()` |
| Varias formas de escribir lo mismo | Consistencia | Diccionario + `replace()` |
| Números con coma decimal | Formato / validez | `.str.replace(",", ".")` + `pd.to_numeric(errors="coerce")` |
| Fechas en varios formatos | Formato | `pd.to_datetime(format=...)` por formato + `fillna()` |
| Unidades mezcladas | Validez | Regla del negocio + `.loc[...]` con la fórmula |
| Dato faltante clave | Completitud | `dropna(subset=[...])` |
| Dato faltante de apoyo | Completitud | `fillna()` con la mediana por grupo (`transform`) |
| Valor imposible | Validez | Reglas con `>`, `<`, `|` y filtro con `~` |
| Valor atípico | (investigar) | Boxplot + regla del IQR |

### 13.3 Tus conclusiones
Escribe como si se lo explicaras a la gerente de la planta, sin términos de programación.

🧠 **Responde:** escribe **tres hallazgos** respaldados por números del cuaderno (máquina, turno, temperatura).

*Tu respuesta:* 
**Tres hallazgos clave del análisis:**

1. **La máquina M-03 es el foco principal de defectos:**
Tras limpiar la nomenclatura y corregir los errores de digitación, la máquina **M-03** representa más del **40% del total de piezas defectuosas** de la planta, registrando la tasa de fallas más alta por lote en comparación con la M-01 y M-02.
2. **El Turno 3 (Noche) concentra la mayor acumulación de fallas:**
El **Turno 3** presenta un promedio de mermas significativamente mayor que los Turnos 1 y 2, concentrando aproximadamente el **45% de los defectos diarios**, lo que evidencia una inestabilidad operativa recurrente durante la jornada nocturna.
3. **Fuerte correlación térmica en M-03 ($r > 0.80$):**
En la M-03, los picos de defectos coinciden de forma directa cuando la temperatura supera el límite operativo seguro de **$28\text{ °C}$** (alcanzando valores críticos de hasta $35\text{--}38\text{ °C}$), a diferencia de la M-01 donde la temperatura se mantiene controlada en su rango óptimo y no muestra correlación directa con las mermas.

🧠 **Responde:** escribe **una recomendación**: ¿a qué máquina le harías mantenimiento primero y por qué?

*Tu respuesta:* 
**Recomendación prioritaria:**

Se debe programar mantenimiento preventivo/correctivo de forma **inmediata a la máquina M-03**.

**¿Por qué?**

* **Sensibilidad térmica y causa raíz de fallas:** La M-03 presenta una alta correlación entre la temperatura y los defectos; cada vez que el equipo se sobrecalienta, la tasa de mermas se dispara. Esto evidencia un fallo directo en su sistema de refrigeración, sellado o calibración térmica.
* **Mayor impacto económico:** Es el equipo que concentra la mayor cantidad absoluta y el mayor porcentaje de piezas defectuosas de toda la planta. Intervenir primero la M-03 generará el mayor impacto positivo en la reducción de mermas y la recuperación de la productividad global.

🧠 **Responde:** ¿qué le propondrías a la planta para que el registro llegue más limpio desde el origen? (Piensa en los errores que encontraste hoy.)

*Tu respuesta:* 
Para evitar repetir las inconsistencias encontradas (registros duplicados, valores negativos, unidades mezcladas, diferencias de caja y errores de tipeo), propondría **cuatro acciones clave para digitalizar y estandarizar la captura de datos**:

1. **Implementar un formulario digital con listas desplegables (Menús Select):**
* **Solución a:** Inconsistencias de texto y nombres de máquinas (`M-03`, ` m-03`, `Maq 1`).
* **Acción:** Restringir la digitación libre en campos categóricos. Las máquinas y turnos solo podrán seleccionarse de una lista predefinida (`M-01`, `M-02`, `M-03` / `Turno 1`, `Turno 2`, `Turno 3`).


2. **Establecer reglas de validación de datos en tiempo real (Límites de entrada):**
* **Solución a:** Valores negativos, cero de más y `defectuosas > producidas`.
* **Acción:** Configurar el sistema para que bloquee el envío si:
* El número de unidades es negativo ($< 0$).
* Las defectuosas superan a las producidas ($Defectuosas > Producidas$).
* La producción ingresada excede la capacidad física máxima por turno (ej. advertencia si ingresan más de $1.500$ unidades).




3. **Estandarizar unidades y aplicar máscaras de formato en los campos:**
* **Solución a:** Mezcla de formatos de fecha y escalas de temperatura (°C vs. °F).
* **Acción:** Forzar un formato de fecha único (`AAAA-MM-DD` mediante un selector de calendario) y fijar la unidad de temperatura exclusivamente a grados Celsius (°C), impidiendo que el operario escriba letras dentro de la casilla numérica.


4. **Automatización mediante lectura de sensores (IoT) e identificación de turno:**
* **Solución a:** Datos faltantes (`NaN`), duplicados y errores humanos de digitación.
* **Acción:** Conectar sensores de temperatura directamente de la máquina a la base de datos para eliminar la toma manual y asociar automáticamente la fecha, hora y operador activo al momento de guardar el registro.

### 13.4 Lista de verificación antes de entregar
- [ ] Ejecuté **todas** las celdas en orden, sin errores (si dudas: *Ejecutar todo*).
- [ ] Mis puntos de control ✅ coinciden con los números indicados.
- [ ] Respondí **todas** las preguntas 🧠, incluidas las tres conclusiones.

### 13.5 Cómo entregar en Moodle
1. Guarda el cuaderno con los resultados visibles.
   - **Colab:** Archivo → Descargar → Descargar .ipynb.
   - **VS Code:** guarda con Ctrl + S; el archivo `.ipynb` ya está en tu carpeta.
2. Entra al aula de **Ciencia de Datos** en Moodle → tarea **"Taller en clase · Semana 9"**.
3. Sube el archivo `.ipynb`. Si trabajaste en pareja, súbelo **cada uno** y escribe el nombre de tu compañero en la primera pregunta 🧠.

¡Buen trabajo! Hoy hiciste lo que más tiempo consume en un proyecto real de datos: dejar los datos listos para confiar en ellos.